# Train GPT mini trên Wikipedia tiếng Việt

Bạn đưa lớp `GPT` tự dựng ở bài Basic 01 vào một vòng train thật: cùng 1 000 bài Wikipedia, cùng tokenizer BPE 8 000 và
cùng cách đo bits-per-character với language model LSTM của module 04 (bài Advanced 02), để hai con số so được với nhau.
Phần được chấm là các mảnh mà mọi vòng pretrain LLM đều có: chia dữ liệu tái lập, lấy batch, learning rate warmup + cosine,
nhóm weight decay, và đánh giá perplexity bằng cửa sổ trượt.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật, train, vẽ; test bỏ qua chúng.
- Mặc định chạy trên CPU (model `n_embd` 128, khoảng 10 phút cả notebook). Có GPU thì chạy bằng kernel `ai-gpu`
  (`setup/gpu`): cell thí nghiệm tự chọn cấu hình lớn hơn (`n_embd` 256, 3 000 bước, đúng ngân sách của LSTM module 04).
- Chấm: `uv run pytest tracks/ai-engineer/modules/05-transformers-llm-architecture/exercises/intermediate/01-train-gpt-wikipedia -q`

In [ ]:
# explore: giới hạn luồng, tìm gốc repo, kiểm Wikipedia tiếng Việt (117 MB, tải một lần bằng env gốc)
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
WIKI = ROOT / "data/raw/wikipedia-vi"
if not (WIKI.is_dir() and any(WIKI.iterdir())):
    sys.path.insert(0, str(ROOT))
    from data.download_dataset import ensure

    ensure("wikipedia-vi")
SHARD = WIKI / "20231101.vi/train-00002-of-00004.parquet"
print(SHARD, SHARD.exists())

In [ ]:
import math
import time
import unicodedata

import numpy as np
import pyarrow.parquet as pq
import torch
import torch.nn.functional as F
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers

## 1. Dữ liệu: đúng cách chia của module 04

Để so với LSTM của module 04, dùng **đúng** tập train và validation của bài đó:

- `long_articles(path, min_chars)`: đọc cột `text` của tệp parquet, giữ bài có `len(text) >= min_chars` (độ dài **trước**
  chuẩn hoá), chuẩn hoá NFC, giữ thứ tự trong tệp.
- `split_articles(articles, n_train, n_val, seed)`: hoán vị bằng `np.random.RandomState(seed).permutation(len(articles))`,
  `n_train` bài đầu của hoán vị là train, `n_val` bài kế tiếp là validation. Trả `(train, val)` là hai list chuỗi.

Văn bản train/validation là các bài nối bằng `"\n\n"`. `train_bpe` (cho sẵn) là BPE byte-level của thư viện `tokenizers`,
giống bài Advanced 02 module 04.

In [ ]:
def long_articles(path, min_chars: int) -> list[str]:
    texts = pq.read_table(path, columns=["text"]).column("text").to_pylist()
    return [unicodedata.normalize("NFC", t) for t in texts if len(t) >= min_chars]


def split_articles(articles: list[str], n_train: int, n_val: int, seed: int) -> tuple[list[str], list[str]]:
    perm = np.random.RandomState(seed).permutation(len(articles))
    return [articles[i] for i in perm[:n_train]], [articles[i] for i in perm[n_train:n_train + n_val]]


def train_bpe(text: str, vocab_size: int) -> Tokenizer:
    tok = Tokenizer(models.BPE())
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tok.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(vocab_size=vocab_size, initial_alphabet=pre_tokenizers.ByteLevel.alphabet())
    tok.train_from_iterator([text], trainer)
    return tok

## 2. Batch, learning rate, optimizer

- `get_batch(data, batch_size, block_size, generator)`: `data` là `LongTensor` 1 chiều các token id. Lấy `batch_size` vị
  trí bắt đầu `s` bằng **một** lệnh `torch.randint(len(data) - block_size, (batch_size,), generator=generator)`; trả
  `x[b] = data[s_b : s_b + block_size]`, `y[b] = data[s_b + 1 : s_b + block_size + 1]`, shape `(batch_size, block_size)`.
- `lr_at(step, max_lr, warmup, total, min_lr)`: `step < warmup` thì tăng tuyến tính `max_lr * (step + 1) / warmup`;
  từ `warmup` tới `total` giảm theo cosine từ `max_lr` về `min_lr`
  ($$\text{min} + \frac12(\text{max} - \text{min})(1 + \cos(\pi \cdot \frac{step - warmup}{total - warmup}))$$); `step >= total` thì `min_lr`.
- `configure_optimizer(model, lr, weight_decay)`: `torch.optim.AdamW` với `betas=(0.9, 0.95)`, hai nhóm: tham số có
  `dim() >= 2` (ma trận Linear, Embedding) chịu `weight_decay`, còn lại (bias, LayerNorm) có `weight_decay = 0`. Tham số
  dùng chung (weight tying) chỉ xuất hiện một lần.

In [ ]:
def get_batch(data: torch.Tensor, batch_size: int, block_size: int, generator: torch.Generator):
    starts = torch.randint(len(data) - block_size, (batch_size,), generator=generator)
    x = torch.stack([data[s:s + block_size] for s in starts])
    y = torch.stack([data[s + 1:s + block_size + 1] for s in starts])
    return x, y


def lr_at(step: int, max_lr: float, warmup: int, total: int, min_lr: float) -> float:
    if step < warmup:
        return max_lr * (step + 1) / warmup
    if step >= total:
        return min_lr
    progress = (step - warmup) / (total - warmup)
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))


def configure_optimizer(model: torch.nn.Module, lr: float, weight_decay: float) -> torch.optim.AdamW:
    params = list(model.parameters())
    decay = [p for p in params if p.dim() >= 2]
    no_decay = [p for p in params if p.dim() < 2]
    groups = [{"params": decay, "weight_decay": weight_decay}, {"params": no_decay, "weight_decay": 0.0}]
    return torch.optim.AdamW(groups, lr=lr, betas=(0.9, 0.95))

## 3. Đánh giá: perplexity và bits-per-character bằng cửa sổ trượt

Định nghĩa perplexity, bits per token và bpc giữ nguyên như [mục 7 module 04](../../../../04-nlp-sequence-models/README.md):
tổng NLL (nat) trên **cả** chuỗi validation, chia số token được dự đoán (perplexity) hoặc số ký tự văn bản nhân $$\ln 2$$ (bpc).
LSTM chuyền state qua cả chuỗi; Transformer chỉ nhìn được `block_size` token, nên đánh giá bằng **cửa sổ trượt**:

- `sliding_windows(n_tokens, block_size, stride)`: danh sách `(start, end, n_new)`. Cửa sổ đầu `(0, min(block_size, n_tokens - 1), ...)`;
  cửa sổ sau bắt đầu thêm `stride`; `end = min(start + block_size, n_tokens - 1)`; `n_new` là số target **chưa được tính**
  ở cửa sổ trước (là các target cuối của cửa sổ). Dừng khi đã tính đủ mọi target $$x_1 .. x_{n-1}$$. Mỗi target được tính đúng
  một lần, và (trừ cửa sổ đầu) có ít nhất `block_size - stride` token ngữ cảnh.
- `evaluate_nll(model, ids, block_size, stride, batch_size, device)`: với mỗi cửa sổ, input `ids[start:end]`, target
  `ids[start + 1:end + 1]`, cộng NLL của `n_new` target cuối. Gom các cửa sổ cùng độ dài thành batch. Chạy `model.eval()`,
  không gradient, trả lại chế độ cũ. Trả `(total_nll, n_predicted)` kiểu `(float, int)`; `model(x)` trả `(logits, loss)`.
- `nll_to_metrics(total_nll, n_tokens, n_chars)`: `{"ppl", "bits_per_token", "bpc"}` như module 04.

`stride = block_size // 2` là mặc định của module này (module 10 đo model đã quantize bằng đúng hàm này).

In [ ]:
def sliding_windows(n_tokens: int, block_size: int, stride: int) -> list[tuple[int, int, int]]:
    windows, scored, start = [], 0, 0
    while scored < n_tokens - 1:
        end = min(start + block_size, n_tokens - 1)
        windows.append((start, end, end - scored))
        scored = end
        start += stride
    return windows


@torch.no_grad()
def evaluate_nll(model, ids, block_size: int, stride: int, batch_size: int = 32, device: str = "cpu") -> tuple[float, int]:
    was_training = model.training
    model.eval()
    ids = torch.as_tensor(ids, dtype=torch.long)
    groups: dict[int, list[tuple[int, int, int]]] = {}
    for w in sliding_windows(len(ids), block_size, stride):
        groups.setdefault(w[1] - w[0], []).append(w)
    total, count = 0.0, 0
    for group in groups.values():
        for i in range(0, len(group), batch_size):
            chunk = group[i:i + batch_size]
            x = torch.stack([ids[s:e] for s, e, _ in chunk]).to(device)
            y = torch.stack([ids[s + 1:e + 1] for s, e, _ in chunk]).to(device)
            logits = model(x)[0].float()
            nll = F.cross_entropy(logits.transpose(1, 2), y, reduction="none")
            for row, (_, _, n_new) in enumerate(chunk):
                total += nll[row, -n_new:].sum().item()
                count += n_new
    model.train(was_training)
    return total, count


def nll_to_metrics(total_nll: float, n_tokens: int, n_chars: int) -> dict[str, float]:
    return {"ppl": math.exp(total_nll / n_tokens), "bits_per_token": total_nll / (n_tokens * math.log(2)),
            "bpc": total_nll / (n_chars * math.log(2))}

## 4. Thí nghiệm

Các cell dưới đây không được chấm. Chúng nạp lớp `GPT` **của bạn** từ bài Basic 01 (nếu bài đó chưa xong thì dùng lời giải),
train, đánh giá và sinh văn bản. Ghi lại thời gian và số đo để so với mục 2 phần lý thuyết.

In [ ]:
# explore: dữ liệu và tokenizer (khoảng 15 giây)
articles = long_articles(SHARD, 2000)
train_articles, val_articles = split_articles(articles, 1000, 100, seed=0)
train_text, val_text = "\n\n".join(train_articles), "\n\n".join(val_articles)
print(f"{len(articles)} bài dài | train {len(train_text):,} ký tự | val {len(val_text):,} ký tự")
tok = train_bpe(train_text, 8000)
train_ids = torch.tensor(tok.encode(train_text).ids, dtype=torch.long)
val_ids = torch.tensor(tok.encode(val_text).ids, dtype=torch.long)
print(f"train {len(train_ids):,} token | val {len(val_ids):,} token | {len(val_text) / len(val_ids):.2f} ký tự mỗi token")

In [ ]:
# explore: nạp GPT của bạn (Basic 01), chưa xong thì dùng lời giải
import importlib.util


def load_gpt_module():
    base = ROOT / "tracks/ai-engineer/modules/05-transformers-llm-architecture"
    for path in [base / "exercises/basic/01-gpt-mini/starter/gpt.py", base / "solutions/basic/01-gpt-mini/gpt.py"]:
        spec = importlib.util.spec_from_file_location("gpt_mini", path)
        module = importlib.util.module_from_spec(spec)
        sys.modules["gpt_mini"] = module
        spec.loader.exec_module(module)
        try:
            module.GPT(module.GPTConfig(vocab_size=10, block_size=4, n_layer=1, n_head=1, n_embd=8))(torch.zeros(1, 2, dtype=torch.long))
            print("dùng", path.relative_to(ROOT))
            return module
        except NotImplementedError:
            continue


gpt_mini = load_gpt_module()

In [ ]:
# explore: train (CPU khoảng 8 phút; GPU kernel ai-gpu khoảng 2-3 phút cho cấu hình lớn hơn)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(4)
if DEVICE == "cuda":
    CFG = dict(n_layer=4, n_head=4, n_embd=256, steps=3000)
else:
    CFG = dict(n_layer=4, n_head=4, n_embd=128, steps=1500)
BLOCK, BATCH, MAX_LR, WARMUP = 128, 32, 1e-3, 200

torch.manual_seed(0)
config = gpt_mini.GPTConfig(vocab_size=8000, block_size=BLOCK, n_layer=CFG["n_layer"], n_head=CFG["n_head"], n_embd=CFG["n_embd"])
model = gpt_mini.GPT(config).to(DEVICE)
opt = configure_optimizer(model, MAX_LR, 0.1)
g = torch.Generator().manual_seed(0)
history, start = [], time.perf_counter()
for step in range(CFG["steps"]):
    for group in opt.param_groups:
        group["lr"] = lr_at(step, MAX_LR, WARMUP, CFG["steps"], MAX_LR / 10)
    x, y = get_batch(train_ids, BATCH, BLOCK, g)
    with torch.autocast("cuda", dtype=torch.bfloat16, enabled=DEVICE == "cuda"):
        _, loss = model(x.to(DEVICE), y.to(DEVICE))
    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    history.append(loss.item())
    if step % 250 == 0 or step == CFG["steps"] - 1:
        print(f"bước {step:5d}  loss {loss.item():.3f}  {time.perf_counter() - start:.0f} s")
train_seconds = time.perf_counter() - start
print(f"{model.num_params():,} tham số, {CFG['steps'] * BATCH * BLOCK:,} token đã thấy, {train_seconds / 60:.1f} phút")

In [ ]:
# explore: đánh giá trên validation, so với LSTM của module 04
start = time.perf_counter()
total, n = evaluate_nll(model, val_ids, BLOCK, BLOCK // 2, device=DEVICE)
metrics = nll_to_metrics(total, n, len(val_text))
print(f"GPT mini: perplexity {metrics['ppl']:.1f} | bit/token {metrics['bits_per_token']:.3f} | bpc {metrics['bpc']:.3f}"
      f" ({time.perf_counter() - start:.0f} s)")
print("LSTM 2 tầng 512, BPE 8 000, 3 000 bước (module 04, Advanced 02): perplexity 117.0 | bpc 1.975")

In [ ]:
# explore: loss theo vị trí trong cửa sổ: Transformer dùng ngữ cảnh tới đâu
model.eval()
with torch.no_grad():
    n_win = (len(val_ids) - 1) // BLOCK
    x = val_ids[:n_win * BLOCK].view(n_win, BLOCK)
    y = val_ids[1:n_win * BLOCK + 1].view(n_win, BLOCK)
    per_pos = torch.zeros(BLOCK)
    for i in range(0, n_win, 64):
        logits = model(x[i:i + 64].to(DEVICE))[0].float()
        per_pos += F.cross_entropy(logits.transpose(1, 2), y[i:i + 64].to(DEVICE), reduction="none").sum(0).cpu()
    per_pos /= n_win
for p in [0, 1, 3, 7, 15, 31, 63, 127]:
    print(f"vị trí {p:3d}: NLL {per_pos[p]:.3f}")

import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(history, lw=0.6)
ax[0].set_xlabel("bước"); ax[0].set_ylabel("loss train"); ax[0].set_title("Loss train")
ax[1].plot(per_pos.numpy())
ax[1].set_xscale("log"); ax[1].set_xlabel("vị trí trong cửa sổ"); ax[1].set_ylabel("NLL trung bình"); ax[1].set_title("Loss theo vị trí")
plt.tight_layout(); plt.show()

In [ ]:
# explore: sinh văn bản từ cùng một câu mở đầu
prompt = torch.tensor([tok.encode("Hà Nội là").ids], device=DEVICE)
for temperature, top_k in [(0, None), (0.8, 40), (1.2, None)]:
    out = model.generate(prompt, 50, temperature=temperature, top_k=top_k,
                         generator=torch.Generator(device=DEVICE).manual_seed(0))
    print(f"T={temperature}, top_k={top_k}: {tok.decode(out[0].tolist())}\n")

## 5. Đọc kết quả

So với mục 2 phần lý thuyết. Ba điều cần thấy:

1. So bpc của bạn với LSTM 9.8 triệu tham số của module 04 (1.975): cùng dữ liệu, cùng tokenizer, cùng đơn vị bpc nên so
   được, nhưng chỉ công bằng khi cùng số bước và cỡ batch. Ghi kèm số tham số và ngân sách của bản bạn chạy (CPU hay GPU).
2. Loss theo vị trí: vị trí 0 chỉ có một token ngữ cảnh, vị trí cuối có cả cửa sổ. Đọc trên biểu đồ ngữ cảnh giúp tới đâu.
   Đó là lý do đánh giá bằng cửa sổ trượt (mọi target có ít nhất `block_size // 2` token ngữ cảnh) thay vì cắt khúc rời.
3. Văn bản sinh đúng chính tả, ngữ pháp cục bộ hợp lý, nội dung vẫn vô nghĩa ở mức câu: 1 000 bài là quá ít. Bài Advanced 01
   train trên toàn bộ bài dài của hai shard.

Sau khi chạy hết notebook, làm phần **trắc nghiệm** của bài trong app.